##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [6]:
# FIX CELL (Colab): download + extract UCF50 into /content/UCF50

import os

%cd /content

# 1) Download (resume-safe)
!curl -L -C - -o UCF50.rar https://www.crcv.ucf.edu/data/UCF50.rar

# 2) Install extractor
!apt-get -y install unrar

# 3) Extract
!mkdir -p /content/UCF50
!unrar x -o+ /content/UCF50.rar /content/UCF50/

# 4) Verify
print("UCF50 exists:", os.path.exists("/content/UCF50"))
print("Top-level items in /content/UCF50:")
print(os.listdir("/content/UCF50")[:20])

/content
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
curl: (60) SSL certificate problem: unable to get local issuer certificate
More details here: https://curl.se/docs/sslcerts.html

curl failed to verify the legitimacy of the server and therefore could not
establish a secure connection to it. To learn more about this situation and
how to fix it, please visit the web page mentioned above.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal

Cannot open /content/UCF50.rar
No such file or directory
No files to extract
UCF50 exists: True
Top-level items in /conte

In [5]:
# =========================
# ARTI 560 - Action Recognition (One-Cell Updated Version)
# Classes: Basketball, BenchPress, Biking
# Includes YouTube-link validation with yt-dlp
# =========================

# ---------- 0) Install/check imports ----------
import os, cv2, random, numpy as np, tensorflow as tf, matplotlib.pyplot as plt, subprocess, sys
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, ConvLSTM2D, MaxPooling3D, TimeDistributed, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Ensure yt-dlp is available (for YouTube validation)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yt-dlp"], check=False)

# ---------- 1) Reproducibility ----------
SEED = 23
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ---------- 2) Paths + params ----------
candidates = ["UCF50", "/content/UCF50", "/content/UCF50/UCF50", "/content"]
YOUTUBE_TEST_DIR = "youtube_test_videos"
os.makedirs(YOUTUBE_TEST_DIR, exist_ok=True)

IMAGE_HEIGHT, IMAGE_WIDTH = 64, 64
SEQUENCE_LENGTH = 20
TEST_SIZE = 0.2
EPOCHS = 60
BATCH_SIZE = 8

print("Working directory:", os.getcwd())
print("\nCandidate dataset paths:")
for c in candidates:
    print(f"  {c} -> {os.path.exists(c)}")

# ---------- 3) Find dataset root ----------
def find_dataset_root(base_dir):
    if not base_dir or not os.path.exists(base_dir):
        return None
    d1 = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    if len(d1) >= 10:
        return base_dir
    for d in d1:
        sub = os.path.join(base_dir, d)
        d2 = [x for x in os.listdir(sub) if os.path.isdir(os.path.join(sub, x))]
        if len(d2) >= 10:
            return sub
    return None

EXTRACT_DIR = "/content/UCF50" if os.path.exists("/content/UCF50") else "UCF50"
dataset_root = find_dataset_root(EXTRACT_DIR)

if dataset_root is None:
    raise ValueError("dataset_root is None. Extract UCF50 correctly and set EXTRACT_DIR to the right path.")

all_class_folders = sorted([d for d in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, d))])
print("\nDetected dataset_root:", dataset_root)
print(f"Found {len(all_class_folders)} class folders.")
print("First class folders:", all_class_folders[:25])

# ---------- 4) Choose required 3 classes ----------
required_classes = ["Basketball", "BenchPress", "Biking"]
missing = [c for c in required_classes if c not in all_class_folders]
if missing:
    raise ValueError(f"These required classes are missing in dataset: {missing}")

CLASSES_LIST = required_classes
print("\nUsing classes:", CLASSES_LIST)
for c in CLASSES_LIST:
    print(f"Class check: {c} ->", os.path.isdir(os.path.join(dataset_root, c)))

# ---------- 5) Utility functions ----------
def frames_extraction(video_path, sequence_length=SEQUENCE_LENGTH, image_height=IMAGE_HEIGHT, image_width=IMAGE_WIDTH):
    frames_list = []
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames < sequence_length:
        cap.release()
        return []

    skip = max(total_frames // sequence_length, 1)

    for i in range(sequence_length):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * skip)
        success, frame = cap.read()
        if not success:
            break
        frame = cv2.resize(frame, (image_width, image_height))
        frame = frame / 255.0
        frames_list.append(frame)

    cap.release()
    return frames_list if len(frames_list) == sequence_length else []

def create_dataset(dataset_dir, classes_list):
    features, labels, paths = [], [], []
    for idx, class_name in enumerate(classes_list):
        class_dir = os.path.join(dataset_dir, class_name)
        if not os.path.isdir(class_dir):
            print(f"[WARNING] Missing class folder: {class_dir}")
            continue

        files = os.listdir(class_dir)
        print(f"\nProcessing class '{class_name}' with {len(files)} files...")

        for f in files:
            video_path = os.path.join(class_dir, f)
            frames = frames_extraction(video_path)
            if len(frames) == SEQUENCE_LENGTH:
                features.append(frames)
                labels.append(idx)
                paths.append(video_path)

    return np.asarray(features), np.array(labels), paths

def create_model(seq_len, h, w, n_classes):
    model = Sequential([
        Input(shape=(seq_len, h, w, 3)),
        ConvLSTM2D(16, (3,3), activation='tanh', return_sequences=True),
        MaxPooling3D(pool_size=(1,2,2), padding='same'),
        TimeDistributed(Dropout(0.2)),

        ConvLSTM2D(32, (3,3), activation='tanh', return_sequences=True),
        MaxPooling3D(pool_size=(1,2,2), padding='same'),
        TimeDistributed(Dropout(0.2)),

        ConvLSTM2D(64, (3,3), activation='tanh', return_sequences=False),
        Dropout(0.3),

        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax')
    ])
    return model

# YouTube predict helper (download from link -> predict)
def predict_youtube_video(youtube_url, saved_model_path):
    temp_vid = "temp_video.mp4"

    if os.path.exists(temp_vid):
        os.remove(temp_vid)

    print(f"\nDownloading video: {youtube_url}")
    download_cmd = ["yt-dlp", "-f", "mp4", "-o", temp_vid, youtube_url]
    result = subprocess.run(download_cmd, capture_output=True, text=True)

    if result.returncode != 0 or not os.path.exists(temp_vid):
        print("Download failed.")
        print(result.stderr[:400])
        return

    print("Extracting frames...")
    frames = frames_extraction(temp_vid)
    if len(frames) != SEQUENCE_LENGTH:
        print(f"Could not extract {SEQUENCE_LENGTH} frames from video.")
        return

    features = np.expand_dims(frames, axis=0)

    model_loaded = tf.keras.models.load_model(saved_model_path)
    predicted_probs = model_loaded.predict(features, verbose=0)[0]
    predicted_class_index = int(np.argmax(predicted_probs))
    predicted_class_name = CLASSES_LIST[predicted_class_index]

    print("--- Prediction Results ---")
    for i, class_name in enumerate(CLASSES_LIST):
        print(f"{class_name}: {predicted_probs[i]*100:.2f}%")
    print(f"Final Verdict: {predicted_class_name.upper()}")

# ---------- 6) Build dataset ----------
features, labels, video_paths = create_dataset(dataset_root, CLASSES_LIST)
print("\nfeatures shape:", features.shape)
print("labels shape:", labels.shape)
print("videos used:", len(video_paths))

if len(features) == 0:
    raise ValueError("No samples found. Check class names/folders/video files.")

# ---------- 7) Train/test split ----------
labels_oh = to_categorical(labels, num_classes=len(CLASSES_LIST))
X_train, X_test, y_train, y_test = train_test_split(
    features, labels_oh, test_size=TEST_SIZE, random_state=SEED, shuffle=True, stratify=labels
)
print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

# ---------- 8) Train model ----------
model = create_model(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, len(CLASSES_LIST))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=5, mode='min', restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

# ---------- 9) Evaluate + plots ----------
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Accuracy')
plt.legend()
plt.show()

# ---------- 10) Save model (mandatory naming) ----------
name = "HaneenAlhabib"
save_path = f"{name}_ucf11_model.h5"
model.save(save_path)
print(f"Model saved as: {save_path}")

# ---------- 11) YouTube validation via links ----------
# Replace these with your real links
vid_1 = "https://youtu.be/Yzt7JDOuMhY?si=NZSBKmYw8KbtmDIB"
vid_2 = "https://youtu.be/k7pUUDHHFLE?si=gxWJwRBUvSZPofzx"
vid_3 = "https://youtube.com/shorts/Wbo0IY1tSRo?si=t9scDLKRivtLYvtY"

print("\n--- YouTube Link Validation ---")
print("CLASSES_LIST order:", CLASSES_LIST)
print("Model file:", save_path)

predict_youtube_video(vid_1, save_path)
predict_youtube_video(vid_2, save_path)
predict_youtube_video(vid_3, save_path)

Working directory: /content

Candidate dataset paths:
  UCF50 -> True
  /content/UCF50 -> True
  /content/UCF50/UCF50 -> False
  /content -> True


ValueError: dataset_root is None. Extract UCF50 correctly and set EXTRACT_DIR to the right path.